In [185]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv
import time
import asyncio


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MUSIXMATCH_API_KEY = os.getenv("MUSIXMATCH_API_KEY")

BASE_URL = "https://api.musixmatch.com/ws/1.1"

client = OpenAI(api_key=OPENAI_API_KEY)


In [186]:
def search_tracks(title, artist=None):
    url = f"{BASE_URL}/track.search"

    params = {
        "apikey": MUSIXMATCH_API_KEY,
        "q_track": title,
        "page_size": 10,
        "s_track_rating": "desc"
    }

    if artist:
        params["q_artist"] = artist

    response = requests.get(url, params=params)
    data = response.json()

    body = data.get("message", {}).get("body", {})
    track_list = body.get("track_list", [])

    return track_list


In [187]:
def search_by_lyrics_snippet(snippet):
    url = f"{BASE_URL}/track.search"
    
    params = {
        "apikey": MUSIXMATCH_API_KEY,
        "q_lyrics": snippet,
        "page_size": 10,
        "s_track_rating": "desc"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # Safe extraction
    body = data.get("message", {}).get("body", {})
    
    # Sometimes body is a list
    if isinstance(body, list):
        track_list = body
    else:
        track_list = body.get("track_list", [])
    
    if not track_list:
        return None
    
    # Handle both formats
    first_item = track_list[0]
    track = first_item.get("track", first_item)
    
    return {
        "track_id": track.get("track_id"),
        "track_name": track.get("track_name"),
        "artist_name": track.get("artist_name"),
        "album_name": track.get("album_name"),
        "isrc": track.get("track_isrc"),
        "rating": track.get("track_rating")
    }


In [188]:
def get_lyrics(track_id):
    url = f"{BASE_URL}/track.lyrics.get"

    params = {
        "apikey": MUSIXMATCH_API_KEY,
        "track_id": track_id
    }

    response = requests.get(url, params=params)
    data = response.json()

    message = data.get("message", {})
    body = message.get("body", {})

    # Handle inconsistent Musixmatch responses
    if isinstance(body, list):
        # No lyrics returned
        return None

    lyrics = body.get("lyrics")

    if lyrics and lyrics.get("lyrics_body"):
        return lyrics.get("lyrics_body")

    return None


In [189]:
import json

def ai_extract_song_info(user_input):
    prompt = f"""
    A user entered this text to search for a song:

    "{user_input}"

    The text might be:
    - a full title
    - part of a title
    - artist name
    - a line from the lyrics

    If the text looks like lyrics, determine the most likely song title and artist.

    For worship songs, return the most popular recorded version.

    Respond ONLY in valid JSON:
    {{
        "title": "...",
        "artist": null or "..."
    }}
    """

    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": prompt}]
    )

    content = response.choices[0].message.content.strip()
    content = content.replace("```json", "").replace("```", "")

    return json.loads(content)


In [190]:
def find_song_ai(user_input):
    print("\n🔎 Understanding request with AI...")
    
    # Step 1 — AI extraction
    song_info = ai_extract_song_info(user_input)
    title = song_info.get("title")
    artist = song_info.get("artist")

    print(f"AI Guess → Title: {title} | Artist: {artist}")

    # Step 2 — First search (title + artist)
    tracks = search_tracks(title, artist)

    # Step 3 — Fallback: search by title only
    if not tracks:
        print("🔁 No results with artist, retrying with title only...")
        tracks = search_tracks(title)

    if not tracks:
        print("❌ No tracks found.")
        return None

    # Step 4 — Find most popular track WITH lyrics
    for item in tracks:
        track = item["track"]
        track_id = track["track_id"]

        lyrics = get_lyrics(track_id)

        if lyrics:
            result = {
                "artist": track["artist_name"],
                "title": track["track_name"],
                "album": track["album_name"],
                "isrc": track.get("track_isrc"),
                "rating": track.get("track_rating"),
                "lyrics": lyrics
            }

            print("\n=== Most Popular Match With Lyrics ===")
            print("Artist :", result["artist"])
            print("Title  :", result["title"])
            print("Album  :", result["album"])
            print("ISRC   :", result["isrc"])
            print("Rating :", result["rating"])

            return result

    print("⚠️ Tracks found but no lyrics available (plan limitation or indexing issue).")
    return None


In [191]:
def display_song(result):
    if not result:
        print("No song found.")
        return
    
    print("\n" + "="*50)
    print(f"Title  : {result['title']}")
    print(f"Artist : {result['artist']}")
    print(f"Album  : {result['album']}")
    print(f"ISRC   : {result['isrc']}")
    print(f"Rating : {result['rating']}")
    print("="*50)
    print("\nLyrics:\n")
    print(result['lyrics'])
    print("="*50)


In [192]:


user_input = input("Enter song title, artist, or lyrics: ")

start = time.time()
song_info = ai_extract_song_info(user_input)
print("AI time:", time.time() - start)

start = time.time()
tracks = search_tracks(song_info["title"], song_info["artist"])
print("Search time:", time.time() - start)

start = time.time()
if tracks:
    get_lyrics(tracks[0]["track"]["track_id"])
print("Lyrics time:", time.time() - start)

result = find_song_ai(user_input)
display_song(result)


AI time: 10.634535789489746
Search time: 0.8511183261871338
Lyrics time: 0.5609135627746582

🔎 Understanding request with AI...
AI Guess → Title: Way Maker | Artist: Sinach

=== Most Popular Match With Lyrics ===
Artist : Sinach
Title  : Way Maker
Album  : Way Maker
ISRC   : TCACN1639805
Rating : 75

Title  : Way Maker
Artist : Sinach
Album  : Way Maker
ISRC   : TCACN1639805
Rating : 75

Lyrics:

You are here, moving in our midst
I worship You, I worship You
You are here, working in this place
I worship You, I worship You

You are here, moving in our midst
I worship You, I worship You
You are here, working in this place
I worship You, I worship You

Way maker, miracle worker, promise keeper
Light in the darkness, my God
That is who You are
Way maker, miracle worker, promise keeper
Light in the darkness, my God
That is who You are

You are here, touching every heart
I worship You, I worship You
You are here, healing every heart
I worship You, I worship You

You are here, turning lives a

In [193]:
# =========================
# GET CHORDS FROM result
# =========================
import os
import json
from openai import AsyncOpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

client = AsyncOpenAI(api_key=OPENAI_API_KEY)


async def map_chords_to_lyrics(song: dict) -> dict:
    prompt = f"""
You are a music chords assistant for worship music.

TASK:
Return the most commonly played worship-band chord arrangement
used in live church settings for this song.

IMPORTANT RULES:
- Prefer the chord flow musicians commonly start with live
- Do NOT rotate progressions arbitrarily
- Preserve the starting chord per lyric line
- Use secondary chords (vi, ii) when commonly used
- Do NOT oversimplify to I–IV–V unless the song is traditionally monotone
- Chords must be inferred from worship-band practice, not generic theory

RETURN ONLY VALID JSON. NO MARKDOWN. NO TEXT.

JSON STRUCTURE (placeholders only):
{{
  "title": "{song['title']}",
  "artist": "{song['artist']}",
  "key": "detected musical key",
  "sections": [
    {{
      "name": "Verse / Chorus",
      "lines": [
        {{
          "chords": ["CHORD_1", "CHORD_2", "CHORD_3", "CHORD_4"],
          "lyrics": "Exact lyric line here"
        }}
      ]
    }}
  ]
}}

LYRICS:
{song['lyrics']}
"""


    response = await client.responses.create(
        model=MODEL,
        input=prompt
    )

    raw = (response.output_text or "").strip()
    return json.loads(raw)


def print_chords(song: dict):
    print(f"\n🎵 {song['title']} — {song['artist']}\n")

    for section in song["sections"]:
        print(f"[{section['name']}]")
        for line in section["lines"]:
            print(" ".join(line["chords"]))
            print(line["lyrics"])
        print()


# =========================
# COPY TITLE & ARTIST
# (same source as display_song)
# =========================
song_for_chords = {
    "title": result["title"],
    "artist": result["artist"],
    "lyrics": result["lyrics"]
}

# TOP-LEVEL await (Jupyter-safe)
chord_data = await map_chords_to_lyrics(song_for_chords)
print_chords(chord_data)



🎵 Way Maker — Sinach

[Verse 1]
A F#m D E
You are here, moving in our midst
A F#m D E
I worship You, I worship You
A F#m D E
You are here, working in this place
A F#m D E
I worship You, I worship You

[Chorus]
A E F#m D
Way maker, miracle worker, promise keeper
A E F#m D
Light in the darkness, my God
A E F#m D
That is who You are

[Verse 2]
A F#m D E
You are here, touching every heart
A F#m D E
I worship You, I worship You
A F#m D E
You are here, healing every heart
A F#m D E
I worship You, I worship You

[Bridge]
A E F#m D
You wipe away all tears, You mend the broken heart
A E F#m D
You're the answer to it all, Jesus

[Chorus]
A E F#m D
Way maker, miracle worker, promise keeper
A E F#m D
Light in the darkness, my God
A E F#m D
That is who You are

[Verse 3]
A F#m D E
You are here touching every life
A F#m D E
I worship You, I worship You
A F#m D E
You are here meeting every need
A F#m D E
I worship You, I worship You

